# HRM-Text 交互式对话 Notebok

本文档教你加载训练好的 checkpint 并与模型对话。

## 前置条件

- 一块 80 GB VRAM 的 GPU（H100 / A100）
- 已训练好的 checkpint（位于 `checkpoints/` 目录下）
- 已安装依赖（`pip install -r requirements.txt`）

In [1]:
# ============================================================
# 1. 导入依赖
# ============================================================
import sys
import os
from pathlib import Path
from glob import glob

import torch
import numpy as np

# 确保在仓库根目录运行
if not os.path.exists("simple_inference_engine.py"):
    print("请确保在 HRM-Text 仓库根目录运行此 notebok")
    print("e.g., cd /path/to/HRM-Text && jupyter notebook ...")
else:
    print("✅ 目录检查通过")

from simple_inference_engine import inference_load_checkpoint, inference_generate

✅ 目录检查通过


In [2]:
# ============================================================
# 2. 指定 checkpint 路径
# ============================================================

# 修改这里为你实际的 checkpint 路径
CKPT_PATH = "/root/HRM/HRM-Text/checkpoints/Sampled HLM-torch/HierarchicalReasoningModel armored-vulture"  # ← 修改这里

# 可选参数
CKPT_EPOCH = None   # None = 自动检测最新 epoch
CKPT_USE_EMA = True # 是否使用 EMA 平滑权重（推荐 True）

# 检查路径是否存在
if not os.path.isdir(CKPT_PATH):
    raise FileNotFoundError(f"checkpint 路径不存在: {CKPT_PATH}")

# 列出可用的 epoch
ckpt_files = glob(os.path.join(CKPT_PATH, "fsdp2_epoch_*"))
epochs = sorted(set(int(Path(f).stem.split("_")[-1]) for f in ckpt_files))
print(f"可用的 epoch: {epochs}")
print(f"将加载: CKPT_EPOCH={CKPT_EPOCH if CKPT_EPOCH else '(自动检测最新)'}, CKPT_USE_EMA={CKPT_USE_EMA}")

可用的 epoch: [1, 2, 3, 4]
将加载: CKPT_EPOCH=(自动检测最新), CKPT_USE_EMA=True


In [4]:
# ============================================================
# 3. 加载模型
# ============================================================
# 这一步可能需要 1-2 分钟，会加载 FSDP2 checkpoint 到单 GPU

print("正在加载 checkpint，请稍候...")
ckpt = inference_load_checkpoint(CKPT_PATH, CKPT_EPOCH, CKPT_USE_EMA)
print("✅ 模型加载完成！")
print(f"   tokenizer: {ckpt.tokenizer_info['tokenizer_path']}")
print(f"   condition 映射: {list(ckpt.tokenizer_info['condition_mapping'].keys())}")
print(f"   BOQ: {ckpt.tokenizer_info['boq']}, EOQ: {ckpt.tokenizer_info['eoq']}, EOA: {ckpt.tokenizer_info['eoa']}")

正在加载 checkpint，请稍候...
Detected latest checkpoint epoch: 4
✅ 模型加载完成！
   tokenizer: /root/HRM/data_io/trained_tokenizers/bpe
   condition 映射: ['cot', 'direct', 'noisy', 'synth']
   BOQ: <|im_start|>, EOQ: <|im_end|>, EOA: <|box_end|>


---
## 4. 单轮对话

在下方的单元格中修改 `prompt` 和 `condition`，然后运行。

In [13]:
# ============================================================
# 4. 单轮对话测试
# ============================================================

# --- 可修改的参数 ---
# prompt = "What is the capital of France?"
# prompt = "What is quantum mechanics? Please provide a detailed, complete, systematic, and comprehensive introduction."
# prompt = "Write a short poem about the beauty of nature."
# prompt = "How to make a cake?"
prompt = "Write a python function that calculates the Fibonacci sequence."
# prompt = "My daughter's kitten passed away due to feline infectious peritonitis (FIP). Please give an introduction to this disease, and then write a passage to comfort my daughter."
condition = "direct"         # "direct" = 直接问答, "synth,cot" = 思维链推理
temperature = 0.0             # 0.0 = 确定性解码, >0 = 随机采样
max_context = 3072            # 最大上下文长度（token）
max_tokens = 1024             # 最大生成长度（token）
# -------------------

# 包装为单条 iterator
iterator = iter([(0, (condition, prompt))])

print(f"条件: {condition}")
print(f"输入: {prompt}")
print(f"温度: {temperature}")
print("-" * 50)

for gen_id, generated_text in inference_generate(
    ckpt, iterator, max_context, max_tokens, batch_size=1, temp=temperature
):
    print(generated_text)
print("-" * 50)

条件: direct
输入: Write a python function that calculates the Fibonacci sequence.
温度: 0.0
--------------------------------------------------
def Fibonacci(n):
    if n == 0:
        return 0
    if n == 1:
        return 1
    if n >= 2:
        return Fibonacci(n - 1) + Fibonacci(n - 2)
--------------------------------------------------


---
## 5. 多轮对话（手动输入）

运行下方单元格后，会进入交互模式。在输入框中输入内容并回车，模型会生成回复。

输入 `exit` 或 `quit` 退出。

In [6]:
# ============================================================
# 5. 多轮交互对话（带历史上下文）
# ============================================================
#
# 说明：
#   训练数据是单轮 (BOQ + cond + Q + EOQ + A + EOA) 格式，模型本身没有
#   原生多轮对话能力。这里通过把过往 Q/A 作为纯文本拼到当前 prompt 前面，
#   给模型一个"上下文窗口"以模拟多轮记忆。
#
#   - 每点一次"生成"：history + 当前问题 → 送入模型
#   - 点"清空历史"：丢弃所有过往 Q/A，重新开始
#   - 输出区每次会先清屏再渲染当前完整对话

import ipywidgets as widgets
from IPython.display import display, clear_output

# --- 可修改的参数 ---
DEFAULT_CONDITION = "direct"
DEFAULT_TEMP = 0.0
MAX_CONTEXT = 3072            # 总 token 预算（含拼接历史）
# -------------------

# 多轮对话历史: List[(user_text, assistant_text)]
chat_history: list[tuple[str, str]] = []


def build_prompt_with_history(history, user_input: str) -> str:
    """把历史对话拼进当前 prompt。
    格式：
        User: q1
        Assistant: a1
        User: q2
        Assistant: a2
        User: <current>
    """
    parts = []
    for u, a in history:
        parts.append(f"User: {u}")
        parts.append(f"Assistant: {a}")
    parts.append(f"User: {user_input}")
    parts.append("Assistant:")
    return "\n".join(parts)


def render_history():
    """重绘整个对话区。"""
    clear_output(wait=True)
    if not chat_history:
        print("（暂无历史，输入问题后点击『生成』开始对话）")
        return
    for i, (u, a) in enumerate(chat_history, 1):
        print(f"{'='*60}")
        print(f"🧑 [#{i}] {u}")
        print(f"{'─'*60}")
        print(f"🤖 {a}")
    print(f"{'='*60}")


# 创建 UI 控件
input_box = widgets.Textarea(
    placeholder='输入你的问题，然后点击"生成"...',
    layout=widgets.Layout(width='100%', height='100px')
)
condition_dropdown = widgets.Dropdown(
    options=['direct', 'synth,cot', 'synth', 'cot'],
    value=DEFAULT_CONDITION,
    description='条件:'
)
temp_slider = widgets.FloatSlider(
    value=DEFAULT_TEMP, min=0.0, max=1.5, step=0.1,
    description='温度:',
    layout=widgets.Layout(width='300px')
)
max_tokens_input = widgets.BoundedIntText(
    value=1024, min=64, max=4096, step=64,
    description='最大生成:'
)
generate_btn = widgets.Button(
    description='生成',
    button_style='primary',
    layout=widgets.Layout(width='100px')
)
clear_btn = widgets.Button(
    description='清空历史',
    button_style='warning',
    layout=widgets.Layout(width='120px')
)
output_area = widgets.Output()


def on_generate_clicked(b):
    user_input = input_box.value.strip()
    if not user_input:
        return
    if user_input.lower() in ('exit', 'quit'):
        return

    condition = condition_dropdown.value
    temp = temp_slider.value
    max_gen = max_tokens_input.value

    # 拼接历史
    full_prompt = build_prompt_with_history(chat_history, user_input)

    # 检查长度，避免溢出 max_context
    tok_len = ckpt.tokenize_prompt(condition, full_prompt).size
    if tok_len >= MAX_CONTEXT:
        with output_area:
            clear_output(wait=True)
            render_history()
            print(f"\n⚠️  拼接历史后 prompt 长度 {tok_len} ≥ {MAX_CONTEXT}，"
                  f"请点『清空历史』或缩短输入。")
        return

    # 调用模型
    iterator = iter([(0, (condition, full_prompt))])
    response = ""
    for _gen_id, generated_text in inference_generate(
        ckpt, iterator,
        max_tokens=MAX_CONTEXT, max_generation=max_gen,
        batch_size=1, temp=temp,
    ):
        response = generated_text

    # 写入历史
    chat_history.append((user_input, response))
    input_box.value = ""

    # 重绘
    with output_area:
        render_history()
        print(f"\n[本轮：cond={condition}, temp={temp}, "
              f"prompt_tok={tok_len}, max_gen={max_gen}]")


def on_clear_clicked(b):
    chat_history.clear()
    input_box.value = ""
    with output_area:
        render_history()


generate_btn.on_click(on_generate_clicked)
clear_btn.on_click(on_clear_clicked)

# 布局
controls = widgets.HBox([condition_dropdown, temp_slider, max_tokens_input,
                         generate_btn, clear_btn])
ui = widgets.VBox([
    widgets.HTML("<h3>HRM-Text 多轮交互对话</h3>"),
    input_box,
    controls,
    output_area,
])

# 初次渲染
with output_area:
    render_history()

display(ui)


---
## 6. 批量测试不同 prompt

如果你想一次性测试多个问题，可以列表形式传入。

In [14]:
# ============================================================
# 6. 批量测试
# ============================================================

# 在这里添加你的测试问题
test_prompts = [
    "What is the capital of France?",
    "If a train travels 120 km in 2 hours, what is its speed?",
    "Explain why the sky is blue.",
    "What is the square root of 144?",
    "What is quantum mechanics? Please provide a detailed, complete, systematic, and comprehensive introduction.",
    "Write a python function that calculates the Fibonacci sequence.",
    "How to make a cup of coffee?",
    "Who was the first president of the United States?",
    "What is the largest planet in our solar system?",
    "What is the chemical symbol for gold?",
    "What is the formula for calculating the area of a circle?",
    "Describe the process of photosynthesis.",
    "What is the difference between AI and machine learning?",
    "What is the tallest mountain in the world?",
    "What is the speed of light?",
    "What is the meaning of life?",
    "Calculate the integral of x^2 from 0 to 1.",
    "Translate 'Hello, how are you?' into French.",
    "Summarize the main points of the following text in bullet points: 'Artificial intelligence (AI) is intelligence demonstrated by machines, in contrast to the natural intelligence displayed by humans and animals.'"
]

condition = "direct"
temperature = 0.0

# 构造 iterator
iterator = iter([(i, (condition, p.strip())) for i, p in enumerate(test_prompts)])

results = [""] * len(test_prompts)
for gen_id, generated_text in inference_generate(
    ckpt, iterator, max_tokens=3072, max_generation=512,
    batch_size=len(test_prompts), temp=temperature
):
    results[gen_id] = generated_text

# 输出结果
for i, (prompt, result) in enumerate(zip(test_prompts, results)):
    print(f"\n{'='*60}")
    print(f"问题 #{i+1}: {prompt}")
    print(f"{'─'*60}")
    print(f"回答: {result}")
print(f"\n{'='*60}")


问题 #1: What is the capital of France?
────────────────────────────────────────────────────────────
回答: Paris

问题 #2: If a train travels 120 km in 2 hours, what is its speed?
────────────────────────────────────────────────────────────
回答: 60 km/h

问题 #3: Explain why the sky is blue.
────────────────────────────────────────────────────────────
回答: The sky is blue because of the scattering of sunlight by water droplets in the atmosphere.

问题 #4: What is the square root of 144?
────────────────────────────────────────────────────────────
回答: 12

问题 #5: What is quantum mechanics? Please provide a detailed, complete, systematic, and comprehensive introduction.
────────────────────────────────────────────────────────────
回答: Quantum mechanics is a branch of physics that describes the behavior of matter and energy at the atomic and subatomic levels. It is based on the principles of quantum mechanics, which was developed by scientists such as Niels Bohr, Max Born, and Richard Feynman. Quantum

---
## 附录：parameter 说明

### condition

控制模型的推理模式。可组合多个条件，用逗号分隔：

| 条件 | 效果 |
|------|------|
| `direct` | 直接回答，不附加推理过程 |
| `synth` | 合成式思维（综合信息后输出） |
| `cot` | Chain-of-Thought 逐步推理 |
| `synth,cot` | 先推理再综合（评测默认配置） |

### temperature

- `0.0` — 确定性解码，每次输出相同（适合评测）
- `0.1~0.7` — 适度随机性，平衡多样性与质量
- `0.8~1.5` — 高随机性，适合创意生成

### batch_size

批量测试时增大 batch_size 可提高吞吐量。需要确保 GPU 显存充足。

---

**注意：** 所有生成代码的 GPU 运行时，建议在终端用 `nvidia-smi` 监控显存使用。